## Problema de cobertura

In [ ]:
%pip install pulp -q

import pulp

prob = pulp.LpProblem("Cobertura", pulp.LpMinimize)

# Uma empresa precisa instalar centros de distribuicao para atender 6 regioes (R1..R6).
# Ha 6 locais candidatos (L1..L6), cada um com custo de instalacao e um conjunto de regioes que cobre.
# Objetivo: minimizar o custo total dos centros instalados garantindo que toda regiao seja coberta
# por pelo menos um centro.

# Variaveis binarias: 1 se o local i for escolhido, 0 caso contrario
x1 = pulp.LpVariable("L1", cat='Binary')
x2 = pulp.LpVariable("L2", cat='Binary')
x3 = pulp.LpVariable("L3", cat='Binary')
x4 = pulp.LpVariable("L4", cat='Binary')
x5 = pulp.LpVariable("L5", cat='Binary')
x6 = pulp.LpVariable("L6", cat='Binary')

# Custos de instalacao (em milhares de R$)
custos = [40, 55, 35, 45, 50, 30]

## RESTRICOES
# Cobertura de cada regiao: pelo menos um local escolhido deve cobri-la
# L1 cobre: R1, R2
# L2 cobre: R1, R3, R5
# L3 cobre: R2, R4, R5
# L4 cobre: R3, R6
# L5 cobre: R1, R4, R6
# L6 cobre: R2, R5, R6
prob += x1 + x2 + x5 >= 1                # R1
prob += x1 + x3 + x6 >= 1                # R2
prob += x2 + x4 >= 1                     # R3
prob += x3 + x5 >= 1                     # R4
prob += x2 + x3 + x6 >= 1                # R5
prob += x4 + x5 + x6 >= 1                # R6

# Funcao obj: minimizar o custo total de instalacao
prob += 40*x1 + 55*x2 + 35*x3 + 45*x4 + 50*x5 + 30*x6

prob.solve(pulp.PULP_CBC_CMD(msg=False))

# Pegando os resultados
escolha = [x1.varValue, x2.varValue, x3.varValue, x4.varValue, x5.varValue, x6.varValue]
custo_total = sum(escolha[i] * custos[i] for i in range(6))

# Cobertura conferida por regiao
cobre = {
    'R1': [1, 2, 5],
    'R2': [1, 3, 6],
    'R3': [2, 4],
    'R4': [3, 5],
    'R5': [2, 3, 6],
    'R6': [4, 5, 6],
}

print("\nPROBLEMA DE COBERTURA - CUSTO MÍNIMO\n")
print(f"Custo Total Mínimo: R$ {custo_total*1000:.2f}\n")
print("Locais Selecionados:")
print("Local\tEscolhido\tCusto (mil R$)")
for i in range(6):
  marca = 'sim' if escolha[i] == 1 else 'não'
  print(f"L{i+1}\t{marca}\t\t{custos[i]}")

print("\nVerificação de Cobertura por Região:")
print("Região\tCoberta por\t\tAtendida?")
for r, locais in cobre.items():
  selec = [f"L{j}" for j in locais if escolha[j-1] == 1]
  status = '✔' if selec else '✖'
  print(f"{r}\t{', '.join(selec) if selec else '-'}\t\t{status}")